# Generate COMPASS platinum-arm figures (PROFILE_data_processing sources)

Identical to `COMPASS_generate_figures.ipynb` except for the data root: this
notebook renders the run built from the **merged PROFILE_data_processing
parquets** (`COMPASS_run_locally_profile_data.ipynb`), reading
`data/CAIA/COMPASS_PROFILE_DATA/` and writing
`figures/CAIA/COMPASS_PROFILE_DATA/`.

Nothing under `data/CAIA/COMPASS/` or `figures/CAIA/COMPASS/` is written, so
the two figure sets can be compared side by side.

Canonical R figure workflow using tidyverse/ggplot2.
**Run All** renders the two ADT-entry ICD-C61 cohort arms: **ARPI** and **ADT**.

Outputs are split by cohort arm, then flattened within each figure directory:

`FIG_ROOT/<ARPI|ADT>/<figure>/<cohort>_<plot-stem>.png`

For example, `figure1/adt_figure1a_consort.png` is written directly in `figure1/`.
Plot PDFs and legacy `original_*` Figure 2 variants are not generated. Tables
remain CSV/Markdown and use the same flat figure/cohort-prefix convention.
There is no `R/` language subdirectory.

All figure-generation logic lives in `COMPASS_generate_figures_pipeline.R` as
`generate_figures(cohort, ...)`, which is shared verbatim with the baseline
notebook -- it is already parameterized on `nepc_proj_path` / `fig_root`, so
there is no PROFILE-specific figure code anywhere.

Run `COMPASS_run_locally_profile_data.ipynb` to completion first; this notebook
reads only that run's outputs.


## Setup


In [ ]:
pipeline_path <- normalizePath(
  file.path(getwd(), "COMPASS_generate_figures_pipeline.R"),
  mustWork = FALSE
)
if (!file.exists(pipeline_path)) {
  pipeline_path <- normalizePath(
    file.path(getwd(), "COMPASS", "survival_analysis", "COMPASS_generate_figures_pipeline.R"),
    mustWork = FALSE
  )
}
if (!file.exists(pipeline_path))
  stop("Could not locate COMPASS_generate_figures_pipeline.R")
source(pipeline_path)

# Run built from the merged PROFILE_data_processing parquets. The baseline
# ALL_2025_03 run stays at .../CAIA/COMPASS and .../figures/CAIA/COMPASS and is
# never written to from this notebook.
NEPC_PROJ_PATH <- "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS_PROFILE_DATA"
FIG_ROOT <- "/data/gusev/USERS/jpconnor/figures/CAIA/COMPASS_PROFILE_DATA"
BASELINE_PROJ_PATH <- "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS"

# generate_figures() reads LLM_NEPC_labels/{baca_lab_annotations.csv,
# LLM_v3_labels.tsv} from NEPC_PROJ_PATH. Those are hand-curated annotation
# INPUTS, not pipeline outputs, so they exist only under the baseline root.
# Link rather than copy, so both runs always see the same annotations.
llm_label_dir <- file.path(NEPC_PROJ_PATH, "LLM_NEPC_labels")
if (!dir.exists(llm_label_dir)) {
  baseline_llm_dir <- file.path(BASELINE_PROJ_PATH, "LLM_NEPC_labels")
  if (!dir.exists(baseline_llm_dir))
    stop("LLM annotations not found at ", baseline_llm_dir,
         " -- generate_figures() cannot run without them.")
  dir.create(NEPC_PROJ_PATH, recursive = TRUE, showWarnings = FALSE)
  if (!file.symlink(baseline_llm_dir, llm_label_dir))
    stop("Could not link ", baseline_llm_dir, " -> ", llm_label_dir,
         " -- copy the directory manually and re-run this cell.")
  message("Linked ", llm_label_dir, " -> ", baseline_llm_dir)
}

# Fail early with a useful message rather than deep inside generate_figures().
for (required_path in c(
  file.path(NEPC_PROJ_PATH, "mrn_lists", "platinum_MRN_list.csv"),
  file.path(NEPC_PROJ_PATH, "mrn_lists", "icd_prostate_mrn_flags.csv"),
  file.path(NEPC_PROJ_PATH, "longitudinal_prediction_data.csv"),
  file.path(NEPC_PROJ_PATH, "longitudinal_prediction_data_adt.csv")
)) {
  if (!file.exists(required_path))
    stop(required_path, " not found -- run COMPASS_run_locally_profile_data.ipynb first.")
}

message("data root:   ", NEPC_PROJ_PATH)
message("figure root: ", FIG_ROOT)

# Generate lab-specific panels only for PSA and Testosterone.
# Non-androgen labs remain available to models and aggregate feature plots.
PLOT_NON_ANDROGEN_DISTRIBUTIONS <- FALSE
PLOT_NON_ANDROGEN_LAB_FIGURES <- FALSE


## Render the reported cohorts

Runs the R figure-generation pipeline for `arpi` and `adt`. Outputs are written under
`FIG_ROOT/ARPI/` and `FIG_ROOT/ADT/`, respectively.


In [ ]:
for (COHORT in COHORTS) {
  message(sprintf("\n========== R figures: %s ==========", COHORT_LABELS[[COHORT]]))
  generate_figures(
    COHORT,
    nepc_proj_path = NEPC_PROJ_PATH,
    fig_root = FIG_ROOT,
    plot_non_androgen_distributions = PLOT_NON_ANDROGEN_DISTRIBUTIONS,
    plot_non_androgen_lab_figures = PLOT_NON_ANDROGEN_LAB_FIGURES
  )
  message(sprintf("\nCompleted R figures for %s.", COHORT_LABELS[[COHORT]]))
}
